# Bias-Variance Tradeoff

We will investigate the <span style="color:#1f77b4">**bias**</span>-<span style="color:#ff7f0e">**variance**</span> tradeoff using a simulated example. 

Recall that 
- <span style="color:#1f77b4">**Bias**</span>: patterns in the mapping ($X \rightarrow y$) failed to be captured by an underparameterized model 
- <span style="color:#ff7f0e">**Variance**</span>: unreliable estimates of the true parameters due to overparameterization, given finite samples

We will generate data from the following model
$$\mathbb{E}[y \mid x] \;=\; \sin (\pi x ) \;+\; 0.5\,\sin (2\pi x ).$$

Try this: 
- Run the following cell to invoke the <span style="color:#2ca02c">**interactive plot**</span>.  
- Change the type of basis functions and the number of basis functions.  
- Explain your observations. 


### How it works

Conceptual algorithm (measuring bias and variance by simulation, when the true curve $f$ is known):

1. Pick a model family with a flexibility knob $K$ (here: the number of basis functions).
2. Fit the model to a training set by least squares on the $K$ basis functions.
3. <span style="color:#1f77b4">**Bias²**</span>: refit on many training sets and average the fitted curves; bias² is the squared gap between that average curve and the true $f$ --- the part of $f$ the family cannot reach.
4. <span style="color:#ff7f0e">**Variance**</span>: how much an individual fitted curve scatters around that average, across training sets --- the part driven by the luck of the noise.
5. Expected test error $=$ bias² $+$ variance $+$ <span style="color:#2ca02c">**irreducible noise**</span> $\sigma^2$. Sweep $K$: bias² falls while variance climbs --- the best $K$ balances the two.


In [1]:
# ----- imports (ipywidgets powers the interactive plot below) ---------------
import numpy as np, matplotlib.pyplot as plt, seaborn as sns
from ipywidgets import Button, Output, VBox
from IPython.display import display
from matplotlib.patches import Rectangle


In [ ]:
sns.set_theme()
from scipy.interpolate import BSpline
from ipywidgets import Dropdown, IntSlider, Output, VBox, HBox
from IPython.display import display

# ------------------------------------------------------------------
# fixed design & true signal
# Fixed design and known truth: x_grid never changes, and f_true is the
# target curve. Knowing the truth is what makes bias measurable at all.
n = 100
x_grid = np.linspace(0, 4, n)
f_true = np.sin(np.pi * x_grid) + 0.5 * np.sin(2 * np.pi * x_grid)
sigma2 = 1.0                      # Var[ε]

# ------------------------------------------------------------------
# Steps 1-2: the flexibility knob. design_matrix(x, K, basis) builds the K
# basis functions; least squares on these columns is the model fit (Step 2).
def design_matrix(x, K, basis):
    x = np.asarray(x)
    if basis == "Polynomial":
        cols = [x**k for k in range(1, K + 1)]
    elif basis == "Trigonometric":
        cols = [np.sin(j * np.pi * x / 2) for j in range(1, K + 1)]
        cols += [np.cos(j * np.pi * x / 2) for j in range(1, K + 1)]
    elif basis.startswith("B-spline"):
        deg = 1 if "linear" in basis else 3
        # uniform internal knots so ~K basis functions
        n_knots = K + deg + 1
        t = np.r_[[x[0]] * (deg+1),
                   np.linspace(x[0], x[-1], n_knots - (deg+1)*2 + 2)[1:-1],
                   [x[-1]] * (deg+1)]
        cols = [BSpline(t, (np.arange(len(t)-deg-1)==i).astype(float), deg)(x)
                for i in range(len(t)-deg-1)]
    else:
        raise ValueError("Unknown basis")
    return np.column_stack(cols)

# Steps 3-5 computed by formula instead of brute-force refitting
# (H is the "hat" matrix that maps observed y to fitted values):
#   bias2    = mean of ((I - H) f_true)^2  -- what the family cannot reach (Step 3)
#   variance = sigma2 * trace(H H^T) / n   -- sensitivity to the noise draw (Step 4)
#   total    = bias2 + variance            -- risk without the constant sigma2 (Step 5)
def risk_components(K, basis):
    X = design_matrix(x_grid, K, basis)
    H = X @ np.linalg.pinv(X.T @ X) @ X.T
    bias2 = np.mean(((np.eye(n) - H) @ f_true) ** 2)
    variance = sigma2 * np.trace(H @ H.T) / n
    return bias2, variance, bias2 + variance

# ------------------------------------------------------------------
# widgets
# What the controls teach:
#   Basis:   different families reach the truth at different K
#   K basis: slide K and watch bias2 fall while variance climbs (Step 5)
basis_dd = Dropdown(
    options=["Polynomial", "Trigonometric",
             "B-spline (linear)", "B-spline (cubic)"],
    value="Polynomial", description="Basis:"
)
K_slider = IntSlider(value=3, min=1, max=20, step=1,
                     description="K basis:", continuous_update=False)
out = Output()

def redraw(*_):
    basis = basis_dd.value
    K_sel = K_slider.value
    Ks = np.arange(1, K_slider.max + 1)
    bias, var, tot = zip(*(risk_components(k, basis) for k in Ks))
    # fitted curve for selected K
    X_sel = design_matrix(x_grid, K_sel, basis)
    # Note: the response used below is the NOISELESS f_true, so the curve shown
    # is the AVERAGE fit over training sets (Step 3's curve): the top panel
    # displays bias only; the noise-driven scatter appears in the bottom panel.
    y_hat = X_sel @ np.linalg.pinv(X_sel.T @ X_sel) @ X_sel.T @ f_true

    with out:
        out.clear_output(wait=True)
        fig, axes = plt.subplots(2, 1, figsize=(7, 7),
                                 height_ratios=[2, 1.4])

        # top: true vs fitted
        axes[0].plot(x_grid, f_true, "k--", label="true f(x)")
        axes[0].plot(x_grid, y_hat, label=f"fit (K={K_sel})")
        axes[0].set(xlabel="x", ylabel="y",
                    title=f"{basis} basis  •  K = {K_sel}")
        axes[0].legend(); axes[0].grid(alpha=.3)

        # bottom: bias², variance, total
        axes[1].plot(Ks, bias, label="bias²")
        axes[1].plot(Ks, var,  label="variance")
        axes[1].plot(Ks, tot,  color="black", label="total risk")
        axes[1].scatter(K_sel, tot[K_sel-1], color="red", zorder=5)
        axes[1].set(xlabel="K", ylabel="Risk"); axes[1].grid(alpha=.3)
        axes[1].legend()

        plt.tight_layout()
        display(fig)      # render inside Output widget
        plt.close(fig)    # prevent duplicate outside widget

# callbacks
basis_dd.observe(redraw, names="value")
K_slider.observe(redraw, names="value")

display(VBox([HBox([basis_dd, K_slider]), out]))
redraw()   # initial draw
